# Проверка заранее заданных гипотез

В работе заранее сформулированы две проверяемые гипотезы. Их формулировки и итог
зафиксированы в препринте (Введение, Результаты, Обсуждение), здесь они собраны в
одном месте и проверяются на данных.

1. Клиническая. Прием алкоголя за сутки до эпизода связан с рецидивирующим течением
   ДКА. Проверяем по протоколу для долей: критерий хи-квадрат Пирсона (при ожидаемых
   частотах больше 10) с отношением шансов и 95% доверительным интервалом.
2. Методологическая. Более сложные модели (случайный лес, градиентный бустинг) дают
   более высокую разделяющую способность, чем логистическая регрессия. Проверяем
   парным критерием DeLong с поправкой Бенджамини-Хохберга, сравнивая все три сложные
   семейства с логистической регрессией на наборе без мультиколлинеарности, на
   out-of-fold обучающей части и на отложенном тесте.

## Гипотеза 1. Прием алкоголя и рецидив

Нулевая гипотеза: связи нет, отношение шансов равно 1. Альтернативная: связь есть,
отношение шансов отличается от 1. Тест выбираем по статистическому протоколу проекта
(`src/stats.py`).

In [ ]:
import sys, pathlib, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display
from src import io, stats

df = io.load_processed()
alc = [c for c in df.columns if 'лкогол' in c][0]
tgt = [c for c in df.columns if 'ецидив' in c and df[c].notna().sum() == 273][0]

sub = df[[alc, tgt]].dropna()
sub = sub[sub[alc].isin([0, 1])]
rec, nor = sub[sub[tgt] == 1], sub[sub[tgt] == 0]

# четырехпольная таблица: строки алкоголь да/нет, столбцы рецидив да/нет
a = int(((sub[alc] == 1) & (sub[tgt] == 1)).sum())
b = int(((sub[alc] == 1) & (sub[tgt] == 0)).sum())
c = int(((sub[alc] == 0) & (sub[tgt] == 1)).sum())
d = int(((sub[alc] == 0) & (sub[tgt] == 0)).sum())
table = np.array([[a, b], [c, d]])

_, _, _, expected = chi2_contingency(table, correction=False)
cmp = stats.compare_proportions(table)
orr = stats.odds_ratio(table)

print(f'Известен алкоголь у {len(sub)} из {int(df[tgt].notna().sum())} пациентов')
print(f'Доля принимавших алкоголь накануне:')
print(f'  рецидив      {(rec[alc] == 1).mean():.3f}  ({int((rec[alc] == 1).sum())}/{len(rec)})')
print(f'  без рецидива {(nor[alc] == 1).mean():.3f}  ({int((nor[alc] == 1).sum())}/{len(nor)})')
print(f'Минимальная ожидаемая частота: {expected.min():.1f}  (больше 10 -> хи-квадрат Пирсона)')
print(f'{cmp.test}: статистика {cmp.statistic:.3f}, p = {cmp.pvalue:.4f}')
print(f'Отношение шансов = {orr["odds_ratio"]:.2f}, 95% ДИ [{orr["ci_low"]:.2f}; {orr["ci_high"]:.2f}]')

display(pd.DataFrame(table, index=['алкоголь: да', 'алкоголь: нет'],
                     columns=['рецидив', 'без рецидива']))

Вывод. Прием алкоголя за сутки до эпизода принимали 29.1% пациентов с рецидивом
против 15.3% в группе единичного эпизода. Минимальная ожидаемая частота больше 10,
поэтому по протоколу применен критерий хи-квадрат Пирсона: p = 0.008. Нулевую гипотезу
отклоняем. Отношение шансов 2.27 (95% ДИ 1.23-4.20): прием алкоголя накануне повышает
шансы рецидива примерно вдвое. После поправки Бенджамини-Хохберга в Таблице 1 связь
сохраняет значимость (q = 0.023). Гипотеза подтверждена и согласуется с литературой.

## Гипотеза 2. Сложность модели и разделяющая способность

Нулевая гипотеза: разделяющая способность сложных моделей не отличается от
логистической регрессии. Альтернативная: сложные модели выше. Проверяем не на одной
выгодной паре, а на всех трех сложных семействах (случайный лес, XGBoost, CatBoost)
против логистической регрессии на наборе без мультиколлинеарности. Сравнение парным
критерием DeLong для связанных ROC-кривых (предсказания спарены по пациентам) с
поправкой Бенджамини-Хохберга, отдельно на out-of-fold обучающей части и на отложенном
тесте. Тот же модуль дополнительно считает попарный DeLong между всеми восемью
итоговыми моделями (28 пар), чтобы проверить, можно ли вообще выделить лучшую. Движок
- `src/model_compare.py` (DeLong и бутстрэп из `src/hypothesis_test.py`).

In [ ]:
from src import config, model_compare

# Считает попарный DeLong между всеми восемью моделями (28 пар) на OOF и тесте
# с поправкой Бенджамини-Хохберга и гипотезу о сложности (3 семейства против
# логрега на наборе без мультиколлинеарности). Печатает сводку, пишет таблицы.
res = model_compare.run()

print('\nТаблица гипотезы о сложности (Таблица 4 препринта):')
h2 = pd.read_csv(config.TABLES_DIR / 'hypothesis_complexity.csv')
display(h2)

Вывод. На обучающей части (OOF) все три сложные модели номинально выше логистической
регрессии (случайный лес 0.772, XGBoost 0.761, CatBoost 0.764 против 0.719), но после
поправки Бенджамини-Хохберга различие незначимо ни у одной (DeLong p = 0.09). На
отложенном тесте порядок обратный: логистическая регрессия выше всех трех (0.802),
различия тоже незначимы. Попарный DeLong между всеми восемью моделями не выявил ни
одной значимой пары (0 из 28) ни на OOF, ни на тесте. Гипотезу о превосходстве сложных
моделей отклоняем: устойчивого преимущества по разделяющей способности на этой выборке
нет, и единую лучшую модель выделить нельзя.

## Итог

Из двух заранее заданных гипотез клиническая (связь алкоголя с рецидивом) подтвердилась,
методологическая (превосходство сложных моделей над логистической регрессией) отклонена.
Это согласуется с общим выводом работы: набор факторов риска переносится на нашу
популяцию, а единую лучшую модель выделить нельзя, поэтому в сервис вынесены все восемь
моделей с выбором за врачом.